# Motorsports Data Analysis Starter

This notebook demonstrates the core data manipulation and visualization tools available in this environment.

## Libraries Included

- **pandas** - Tabular data manipulation
- **numpy** - Numeric computing
- **pyarrow** - Efficient columnar data format (Parquet)
- **matplotlib** - Static plotting
- **seaborn** - Statistical visualization
- **plotly** - Interactive plots

In [ ]:
# Import core libraries
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Configure matplotlib for better inline display
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"pyarrow version: {pa.__version__}")

## Example 1: Creating Sample Lap Data

Let's create a sample dataset that resembles typical motorsports telemetry data.

In [ ]:
# Generate sample lap data
np.random.seed(42)
n_laps = 20

lap_data = pd.DataFrame({
    'lap': np.arange(1, n_laps + 1),
    'lap_time_s': np.random.normal(loc=92.5, scale=1.8, size=n_laps).round(3),
    'max_speed_kph': np.random.normal(loc=182, scale=4, size=n_laps).round(1),
    'min_speed_kph': np.random.normal(loc=65, scale=3, size=n_laps).round(1),
    'avg_throttle_pct': np.random.normal(loc=68, scale=5, size=n_laps).round(1),
    'max_brake_bar': np.random.normal(loc=110, scale=8, size=n_laps).round(1),
    'max_lateral_g': np.random.normal(loc=1.85, scale=0.12, size=n_laps).round(2),
})

# Add some variability - simulate tire degradation
lap_data['lap_time_s'] = lap_data['lap_time_s'] + (lap_data['lap'] * 0.05)

# Add session type
lap_data['session'] = ['Practice' if i < 8 else 'Qualifying' if i < 15 else 'Race' 
                        for i in range(n_laps)]

print(f"Generated {len(lap_data)} laps of sample data")
lap_data.head(10)

## Example 2: Basic Data Analysis

Quick statistical overview and session comparison.

In [ ]:
# Overall statistics
print("=== Overall Statistics ===")
print(lap_data[['lap_time_s', 'max_speed_kph', 'max_lateral_g']].describe())

print("\n=== Statistics by Session ===")
session_stats = lap_data.groupby('session').agg({
    'lap_time_s': ['mean', 'min', 'std'],
    'max_speed_kph': 'mean',
    'max_lateral_g': 'mean'
}).round(2)
session_stats

## Example 3: Static Visualization with Matplotlib/Seaborn

Create publication-quality static plots.

In [ ]:
# Create a figure with multiple subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Lap Analysis Dashboard', fontsize=16, fontweight='bold')

# 1. Lap times over session
ax1 = axes[0, 0]
for session in lap_data['session'].unique():
    session_data = lap_data[lap_data['session'] == session]
    ax1.plot(session_data['lap'], session_data['lap_time_s'], 
             marker='o', label=session, linewidth=2)
ax1.set_xlabel('Lap Number', fontsize=11)
ax1.set_ylabel('Lap Time (s)', fontsize=11)
ax1.set_title('Lap Times Progression', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Speed distribution by session
ax2 = axes[0, 1]
sns.boxplot(data=lap_data, x='session', y='max_speed_kph', ax=ax2)
ax2.set_xlabel('Session', fontsize=11)
ax2.set_ylabel('Max Speed (kph)', fontsize=11)
ax2.set_title('Speed Distribution by Session', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# 3. Correlation: Speed vs Lap Time
ax3 = axes[1, 0]
scatter = ax3.scatter(lap_data['max_speed_kph'], lap_data['lap_time_s'], 
                      c=lap_data['lap'], cmap='viridis', s=100, alpha=0.6)
ax3.set_xlabel('Max Speed (kph)', fontsize=11)
ax3.set_ylabel('Lap Time (s)', fontsize=11)
ax3.set_title('Speed vs Lap Time', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax3, label='Lap #')
ax3.grid(True, alpha=0.3)

# 4. G-force and throttle correlation
ax4 = axes[1, 1]
ax4.scatter(lap_data['avg_throttle_pct'], lap_data['max_lateral_g'], 
            c=lap_data.index, cmap='plasma', s=100, alpha=0.6)
ax4.set_xlabel('Avg Throttle (%)', fontsize=11)
ax4.set_ylabel('Max Lateral G', fontsize=11)
ax4.set_title('Throttle vs Lateral G-Force', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Example 4: Interactive Visualization with Plotly

Create interactive plots that allow zooming, panning, and hovering for detailed info.

In [ ]:
# Interactive lap time chart
fig = px.line(lap_data, x='lap', y='lap_time_s', color='session',
              title='Interactive Lap Times by Session',
              labels={'lap': 'Lap Number', 'lap_time_s': 'Lap Time (s)'},
              markers=True,
              hover_data=['max_speed_kph', 'max_lateral_g'])

fig.update_layout(
    hovermode='x unified',
    height=500,
    template='plotly_white'
)

fig.show()

In [ ]:
# Interactive 3D scatter plot
fig = px.scatter_3d(lap_data, 
                    x='max_speed_kph', 
                    y='max_lateral_g', 
                    z='lap_time_s',
                    color='session',
                    size='avg_throttle_pct',
                    hover_data=['lap'],
                    title='3D Performance Analysis',
                    labels={
                        'max_speed_kph': 'Max Speed (kph)',
                        'max_lateral_g': 'Max Lateral G',
                        'lap_time_s': 'Lap Time (s)'
                    })

fig.update_layout(height=600)
fig.show()

## Example 5: Working with Parquet Files

Save and load data efficiently using PyArrow's Parquet format - ideal for large telemetry datasets.

In [ ]:
# Save to Parquet format (compressed by default)
output_file = 'sample_lap_data.parquet'

table = pa.Table.from_pandas(lap_data)
pq.write_table(table, output_file, compression='snappy')

print(f"✓ Saved {len(lap_data)} rows to '{output_file}'")

# Show file size
import os
file_size = os.path.getsize(output_file)
print(f"  File size: {file_size:,} bytes ({file_size/1024:.2f} KB)")

In [ ]:
# Read back from Parquet
loaded_table = pq.read_table(output_file)
loaded_df = loaded_table.to_pandas()

print(f"✓ Loaded {len(loaded_df)} rows from '{output_file}'")
print(f"\nColumns: {list(loaded_df.columns)}")
print(f"\nFirst few rows:")
loaded_df.head()

## Example 6: Advanced Analysis - Rolling Statistics

Calculate moving averages and trends useful for understanding performance over a stint.

In [ ]:
# Calculate rolling averages (3-lap window)
lap_data['lap_time_rolling_avg'] = lap_data['lap_time_s'].rolling(window=3, min_periods=1).mean()
lap_data['speed_rolling_avg'] = lap_data['max_speed_kph'].rolling(window=3, min_periods=1).mean()

# Calculate lap time delta from best
best_lap = lap_data['lap_time_s'].min()
lap_data['delta_to_best'] = lap_data['lap_time_s'] - best_lap

# Plot with rolling average
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(lap_data['lap'], lap_data['lap_time_s'], 
        marker='o', label='Lap Time', linewidth=2, alpha=0.6)
ax.plot(lap_data['lap'], lap_data['lap_time_rolling_avg'], 
        linestyle='--', label='3-Lap Rolling Avg', linewidth=2.5, color='red')

# Mark best lap
best_lap_idx = lap_data['lap_time_s'].idxmin()
ax.scatter(lap_data.loc[best_lap_idx, 'lap'], 
           lap_data.loc[best_lap_idx, 'lap_time_s'],
           s=300, color='gold', marker='*', zorder=5, 
           edgecolors='black', linewidths=2, label='Best Lap')

ax.set_xlabel('Lap Number', fontsize=12)
ax.set_ylabel('Lap Time (s)', fontsize=12)
ax.set_title('Lap Time Progression with Rolling Average', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best lap: Lap {lap_data.loc[best_lap_idx, 'lap']} - {best_lap:.3f}s")

## Next Steps

Now you're ready to work with real AiM logger data:

1. **Export your AiM data** to CSV or another compatible format
2. **Load it** using `pd.read_csv()` or similar
3. **Explore** the columns and data structure
4. **Analyze** using the techniques shown above
5. **Visualize** your actual telemetry data
6. **Save** processed data to Parquet for fast reload

### Suggested Directory Structure

```
motorsports_data_notebook/
├── data/
│   ├── raw/          # Original AiM exports
│   └── processed/    # Cleaned/processed Parquet files
├── notebooks/        # Your analysis notebooks
└── src/              # Custom Python modules (optional)
```

Happy data wrangling! 🏁